# Graph-Based Fraud Detection with GNNs

Companion notebook for the [Graph-Based Fraud Detection wiki page](https://ml-viz-ruby.vercel.app/wiki/graph-fraud-detection).

We build an entity-sharing graph, implement GraphSAGE-style message passing, and simulate fraud propagation to demonstrate guilt-by-association.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

## 1 — Building the entity-sharing graph

In [ ]:
# Simulated entity graph: 12 accounts, 5 shared devices
# (account_i, account_j) share a device
edges = [(0,1), (1,2), (2,3),   # chain: accounts 0-3 share devices
         (5,6), (6,7),           # separate cluster 5-7
         (9,10), (10,11)]        # another cluster

n_accounts = 12
adj = np.zeros((n_accounts, n_accounts))
for i,j in edges:
    adj[i,j] = adj[j,i] = 1.0

# Node features: [transaction_amount_z, account_age_days, n_countries_24h]
features = rng.uniform(0, 1, (n_accounts, 3))

# Known fraud labels (from human review): accounts 0 and 5
known_fraud = {0: 1.0, 5: 1.0}
fraud_scores = np.zeros(n_accounts)
for k, v in known_fraud.items():
    fraud_scores[k] = v

print("Graph adjacency matrix (12x12):")
print(adj.astype(int))
print("\nInitial fraud scores:", fraud_scores)

## 2 — GraphSAGE-style message passing

In [ ]:
def graphsage_mean_agg(h, adj, W_self, W_neigh):
    """
    One layer of GraphSAGE with mean aggregation.
    h: (N, d_in) node features
    adj: (N, N) adjacency (not normalized)
    W_self, W_neigh: (d_in, d_out) weight matrices
    """
    # Normalize adjacency: 1/degree per row
    deg = adj.sum(1, keepdims=True).clip(min=1)
    neigh_mean = (adj / deg) @ h          # mean of neighbors
    # Concatenate self + neighbor and project
    out = np.tanh(h @ W_self + neigh_mean @ W_neigh)
    return out

d_in, d_out = 3, 4
W_self  = rng.normal(0, 0.1, (d_in, d_out))
W_neigh = rng.normal(0, 0.1, (d_in, d_out))

h1 = graphsage_mean_agg(features, adj, W_self, W_neigh)
print(f"After 1 GraphSAGE layer: {features.shape} → {h1.shape}")
print("Node 1 representation (connected to known fraud node 0):", h1[1].round(3))
print("Node 9 representation (connected to fraud cluster 5-7):", h1[9].round(3))

## 3 — Fraud score propagation

In [ ]:
def propagate_fraud_scores(scores, adj, alpha=0.4, n_steps=3):
    """
    Propagate fraud risk through the graph.
    alpha: weight given to neighbor scores vs own score.
    """
    s = scores.copy()
    for step in range(n_steps):
        deg = adj.sum(1).clip(min=1)
        neigh_avg = (adj @ s) / deg
        s = (1 - alpha) * s + alpha * neigh_avg
    return s

propagated = propagate_fraud_scores(fraud_scores, adj)
print("Account | Initial fraud | After graph propagation | Connected to fraud?")
for i in range(n_accounts):
    conn = any(adj[i,j]>0 and fraud_scores[j]>0 for j in range(n_accounts))
    print(f"  {i:3d}   |   {fraud_scores[i]:.3f}       |   {propagated[i]:.3f}                  | {'YES' if conn else 'no'}")

## ✏️ Your turn — label propagation on bipartite graph

In [ ]:
def label_prop_step(scores, adj, n_steps=1):
    """
    Run n_steps of label propagation: each node averages its own score
    with the average of its neighbors' scores.
    Returns updated scores.
    """
    # TODO(you): implement n_steps iterations of the update:
    # s_new[v] = 0.5 * s[v] + 0.5 * mean(s[neighbors of v])
    return ...

prop = label_prop_step(fraud_scores, adj, n_steps=2)
print("After 2-step label propagation:")
for i in range(n_accounts):
    print(f"  Account {i:2d}: {prop[i]:.4f}")

<details><summary>Solution</summary>

```python
def label_prop_step(scores, adj, n_steps=1):
    s = scores.copy()
    for _ in range(n_steps):
        deg = adj.sum(1).clip(min=1)
        s = 0.5 * s + 0.5 * (adj @ s) / deg
    return s
```
</details>